
# 🧠 Human-Like Picker — Train Models
This notebook trains two models from your built dataset:
1) **Move-choice ranker** — pick the most human-like move among Stockfish top-20.
2) **Think-time regressor** — predict human think time (ms) for the chosen move.

**Input**: the parquet built by your Top-20 Fast pipeline, e.g. `data/candidates_top20_fast_parallel_10k.parquet` or the *all* version.


In [1]:
# Install deps (LightGBM for ranking + parquet reader)
%pip -q install pandas pyarrow lightgbm scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:

import os, math, numpy as np, pandas as pd
from pathlib import Path

# Pick your dataset path here (adjust if you wrote the _all version)
DATA_PATH = Path("data/candidates_top20_fast_parallel_10k.parquet")
if not DATA_PATH.exists():
    alt = Path("data/candidates_top20_fast_parallel_10k_all.parquet")
    if alt.exists():
        DATA_PATH = alt

print("Using dataset:", DATA_PATH)
df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()


Using dataset: data\candidates_top20_fast_parallel_10k.parquet
(5590508, 14)


,group_id,fen,side,uci,label_choice,human_move_uci,human_think_ms,in_top20,ply,cp,mate,depth,pv_len,cp_to_best
0,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,e2e4,0,d2d4,0.0,1,1,36,0,0,0,0
1,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,d2d4,1,d2d4,0.0,1,1,20,0,0,0,16
2,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,g1f3,0,d2d4,0.0,1,1,19,0,0,0,17
3,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,e2e3,0,d2d4,0.0,1,1,18,0,0,0,18
4,5850265614594899970:0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,1,c2c4,0,d2d4,0.0,1,1,17,0,0,0,19



## Clean & Features
We avoid label leakage: **do not use** `label_choice`, `human_move_uci`, or `in_top20` as features.  
We keep numeric features and add a compact encoding from `uci`.


In [3]:

import re

# Basic numeric features available
BASE_FEATS = ["side", "cp", "mate", "depth", "pv_len", "cp_to_best", "ply"]

# Encode UCI to compact numeric fields: from_sq (0..63), to_sq (0..63), promo (0..5)
FILES = {c:i for i,c in enumerate("abcdefgh")}
def sq_to_idx(sq):
    # 'a1'->0, 'b1'->1, ..., 'h8'->63
    if not isinstance(sq, str) or len(sq)<2: return -1
    file = FILES.get(sq[0], None)
    rank = int(sq[1]) - 1 if len(sq) > 1 and sq[1].isdigit() else None
    if file is None or rank is None or not (0<=rank<=7): return -1
    return rank*8 + file

PROMO = {"":0, "q":1, "r":2, "b":3, "n":4}
def uci_to_numeric(uci):
    uci = str(uci)
    from_sq = sq_to_idx(uci[0:2])
    to_sq   = sq_to_idx(uci[2:4])
    promo   = PROMO.get(uci[4:], 0) if len(uci) > 4 else 0
    return from_sq, to_sq, promo

fs, ts, ps = [], [], []
for u in df["uci"].astype(str).tolist():
    f, t, p = uci_to_numeric(u)
    fs.append(f); ts.append(t); ps.append(p)
df["uci_from"] = fs
df["uci_to"]   = ts
df["uci_promo"]= ps

FEATS = BASE_FEATS + ["uci_from","uci_to","uci_promo"]
df[FEATS + ["label_choice","in_top20","human_move_uci","human_think_ms"]].head(3)


,side,cp,mate,depth,pv_len,cp_to_best,ply,uci_from,uci_to,uci_promo,label_choice,in_top20,human_move_uci,human_think_ms
0,1,36,0,0,0,0,1,12,28,0,0,1,d2d4,0.0
1,1,20,0,0,0,16,1,11,27,0,1,1,d2d4,0.0
2,1,19,0,0,0,17,1,6,21,0,0,1,d2d4,0.0



## Train/Valid split by **group_id**
We split by groups so no leakage occurs across candidates of the same position.


In [4]:

from sklearn.model_selection import train_test_split

groups = df["group_id"].unique()
g_train, g_valid = train_test_split(groups, test_size=0.15, random_state=42, shuffle=True)

train = df[df["group_id"].isin(g_train)].copy()
valid = df[df["group_id"].isin(g_valid)].copy()

print("Train groups:", train["group_id"].nunique(), "rows:", len(train))
print("Valid groups:", valid["group_id"].nunique(), "rows:", len(valid))


Train groups: 243070 rows: 4751459
Valid groups: 42895 rows: 839049



## 1) Move-choice Ranker (LightGBM LambdaRank)
We train a ranking model to score candidate moves per group. At inference, pick the max-score move.


In [5]:
import lightgbm as lgb

# --- sort by group to align rows with group sizes ---
train_sorted = train.sort_values(["group_id"]).reset_index(drop=True)
valid_sorted = valid.sort_values(["group_id"]).reset_index(drop=True)

X_train = train_sorted[FEATS].fillna(0).astype(float)
y_train = train_sorted["label_choice"].astype(int)
X_valid = valid_sorted[FEATS].fillna(0).astype(float)
y_valid = valid_sorted["label_choice"].astype(int)

# group sizes must follow the same order as data rows
q_train = train_sorted.groupby("group_id").size().astype(int).values
q_valid = valid_sorted.groupby("group_id").size().astype(int).values

lgb_train = lgb.Dataset(X_train, label=y_train, group=q_train, params={"feature_pre_filter": False},)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, group=q_valid, reference=lgb_train)

params = dict(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[1],
    boosting_type="gbdt",
    num_leaves=63,
    learning_rate=0.05,
    min_data_in_leaf=50,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=1,
    max_depth=-1,
    verbose=-1,
)

ranker = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train","valid"],
    num_boost_round=200,
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=20),
    ],
)

# Validation: top-1 accuracy by group
valid_eval = valid_sorted.copy()
valid_eval["pred_score"] = ranker.predict(X_valid, num_iteration=ranker.best_iteration)

top_by_group = (valid_eval
                .sort_values(["group_id","pred_score"], ascending=[True, False])
                .groupby("group_id").head(1))
top1_acc = (top_by_group["label_choice"] == 1).mean()
print(f"Top-1 accuracy (by group): {top1_acc:.4f}")

# Coverage-conditioned accuracy (only groups where a human move exists in top-20)
valid_group_cov = valid_eval.groupby("group_id")["label_choice"].max().reset_index()
covered_groups = set(valid_group_cov[valid_group_cov["label_choice"]==1]["group_id"])
covered_acc = top_by_group[top_by_group["group_id"].isin(covered_groups)]
print(f"Top-1 accuracy on covered groups: {(covered_acc['label_choice']==1).mean():.4f}")


c:\Users\ppava\anaconda3\envs\chess-bot-env\Lib\site-packages\lightgbm\basic.py:2535: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


Training until validation scores don't improve for 30 rounds
[20]	train's ndcg@1: 0.400987	valid's ndcg@1: 0.398741
[40]	train's ndcg@1: 0.411881	valid's ndcg@1: 0.408136
[60]	train's ndcg@1: 0.417542	valid's ndcg@1: 0.414501
[80]	train's ndcg@1: 0.421973	valid's ndcg@1: 0.41886
[100]	train's ndcg@1: 0.425227	valid's ndcg@1: 0.422473
[120]	train's ndcg@1: 0.428115	valid's ndcg@1: 0.424572
[140]	train's ndcg@1: 0.430098	valid's ndcg@1: 0.426973
[160]	train's ndcg@1: 0.432114	valid's ndcg@1: 0.427789
[180]	train's ndcg@1: 0.433859	valid's ndcg@1: 0.429234
[200]	train's ndcg@1: 0.435385	valid's ndcg@1: 0.430143
Did not meet early stopping. Best iteration is:
[200]	train's ndcg@1: 0.435385	valid's ndcg@1: 0.430143
Top-1 accuracy (by group): 0.3962
Top-1 accuracy on covered groups: 0.4101


In [6]:
from lightgbm import LGBMRanker

ranker = LGBMRanker(
    objective="lambdarank",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.9,
    random_state=42,
)
ranker.fit(
    X_train, y_train,
    group=q_train,
    eval_set=[(X_valid, y_valid)],
    eval_group=[q_valid],
    eval_at=[1],
    callbacks=[lgb.early_stopping(60), lgb.log_evaluation(20)],
)


Training until validation scores don't improve for 60 rounds
[20]	valid_0's ndcg@1: 0.39944
[40]	valid_0's ndcg@1: 0.407833
[60]	valid_0's ndcg@1: 0.414943
[80]	valid_0's ndcg@1: 0.418604
[100]	valid_0's ndcg@1: 0.423219
[120]	valid_0's ndcg@1: 0.424921
[140]	valid_0's ndcg@1: 0.427229
[160]	valid_0's ndcg@1: 0.427975
[180]	valid_0's ndcg@1: 0.429211
[200]	valid_0's ndcg@1: 0.429444
Did not meet early stopping. Best iteration is:
[197]	valid_0's ndcg@1: 0.429584


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20



## 2) Think-time Regressor (only on human-chosen rows)
We fit a regressor on the **positive** rows (where `label_choice==1` and think-time is present).  
We predict **log(1 + ms)** for stability; report MAE in ms on validation positives.


In [7]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Filter to positives with think time
train_pos = train[(train["label_choice"]==1) & (train["human_think_ms"].notnull())].copy()
valid_pos = valid[(valid["label_choice"]==1) & (valid["human_think_ms"].notnull())].copy()

def features_df(df_):
    return df_[FEATS].fillna(0).astype(float)

X_tr = features_df(train_pos)
y_tr = np.log1p(train_pos["human_think_ms"].astype(float))

X_va = features_df(valid_pos)
y_va = np.log1p(valid_pos["human_think_ms"].astype(float))

reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)
reg.fit(X_tr, y_tr)

pred_log = reg.predict(X_va)
pred_ms = np.expm1(pred_log).clip(min=0)

mae = mean_absolute_error(np.expm1(y_va), pred_ms)
print(f"Think-time MAE (ms) on validation positives: {mae:.1f} ms")


Think-time MAE (ms) on validation positives: 2422.8 ms



## Save models


In [8]:

import joblib, time
from pathlib import Path

out_dir = Path("models"); out_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")
rank_path = out_dir / f"move_ranker_lgb_{ts}.pkl"
time_path  = out_dir / f"time_regressor_rf_{ts}.pkl"

joblib.dump(ranker, rank_path)
joblib.dump(reg, time_path)

print("Saved:")
print(" -", rank_path)
print(" -", time_path)


Saved:
 - models\move_ranker_lgb_20251126_103155.pkl
 - models\time_regressor_rf_20251126_103155.pkl


In [9]:
import itertools
import random
import lightgbm as lgb

base_params = dict(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[1],
    boosting_type="gbdt",
    verbose=-1,
    feature_pre_filter=False,
)

search_space = {
    "num_leaves":        [31, 63, 127],
    "max_depth":         [-1, 6, 8, 10],
    "min_data_in_leaf":  [20, 50, 100, 200],
    "feature_fraction":  [0.6, 0.8, 1.0],
    "bagging_fraction":  [0.6, 0.8, 1.0],
    "bagging_freq":      [0, 1],
    "learning_rate":     [0.03, 0.05, 0.1],
    "lambda_l1":         [0.0, 0.1, 1.0],
    "lambda_l2":         [0.0, 0.1, 1.0],
}

def sample_params():
    params = base_params.copy()
    for k, v_list in search_space.items():
        params[k] = random.choice(v_list)
    return params

best_score = -1
best_params = None

for trial in range(30):  # e.g. 30 random trials
    params = sample_params()
    print(f"\n=== Trial {trial+1} params ===")
    print(params)

    model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train", "valid"],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)

# For Booster from lgb.train -> use best_score (no underscore)
print(model.best_score)  # good to inspect once to see the keys

valid_score = model.best_score["valid"]["ndcg@1"]
print(f"Valid NDCG@1: {valid_score:.6f}")



=== Trial 1 params ===
{'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [1], 'boosting_type': 'gbdt', 'verbose': -1, 'feature_pre_filter': False, 'num_leaves': 63, 'max_depth': -1, 'min_data_in_leaf': 200, 'feature_fraction': 0.6, 'bagging_fraction': 1.0, 'bagging_freq': 1, 'learning_rate': 0.1, 'lambda_l1': 1.0, 'lambda_l2': 0.1}
Training until validation scores don't improve for 50 rounds
[50]	train's ndcg@1: 0.416851	valid's ndcg@1: 0.412915
[100]	train's ndcg@1: 0.424997	valid's ndcg@1: 0.420585
[150]	train's ndcg@1: 0.433217	valid's ndcg@1: 0.42611
[200]	train's ndcg@1: 0.438096	valid's ndcg@1: 0.428511
[250]	train's ndcg@1: 0.441667	valid's ndcg@1: 0.429934
[300]	train's ndcg@1: 0.445073	valid's ndcg@1: 0.430493
[350]	train's ndcg@1: 0.447406	valid's ndcg@1: 0.431076
[400]	train's ndcg@1: 0.450163	valid's ndcg@1: 0.430866
Early stopping, best iteration is:
[370]	train's ndcg@1: 0.448591	valid's ndcg@1: 0.431356

=== Trial 2 params ===
{'objective': 'lambdarank', 'me